# 1. Dependencies and imports

In [1]:
!pip install -q torch-geometric rdkit py3Dmol fuzzywuzzy
!pip install -q PyTDC==0.4.1 --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 45.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import torch
import torch.nn.functional as F
from torch.nn import Linear

from torch_geometric.utils.smiles import from_smiles
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from tdc.multi_pred import DTI

# check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

Running on: cpu


# 2. Load Davis Dataset

In [3]:
print("Fetching Davis benchmark data...")
data = DTI(name='DAVIS')
df = data.get_data()

# taking a subset of 200 rows
sample_df = df.head(200)

print(f"Loaded {len(sample_df)} drug-target interaction pairs.")
sample_df.head(3)

Downloading...


Fetching Davis benchmark data...


100%|██████████| 21.4M/21.4M [00:00<00:00, 25.3MiB/s]
Loading...
Done!


Loaded 200 drug-target interaction pairs.


,Drug_ID,Drug,Target_ID,Target,Y
0,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,AAK1,MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQV...,43.0
1,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,ABL1p,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0
2,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,ABL2,MVLGTVLLPPNSYGRDQDTSLCCLCTEASESALPDLTDHFASCVED...,10000.0


In [4]:
print("Converting SMILES into PyTorch graph tensors...")
pytorch_graphs = []

for idx, row in sample_df.iterrows():
    smiles = row['Drug']
    affinity = row['Y']

    try:
        graph = from_smiles(smiles)

        # cast node features to float for matrix multiplication
        graph.x = graph.x.float()

        # store binding score as target tensor
        graph.y = torch.tensor([affinity], dtype=torch.float)

        pytorch_graphs.append(graph)
    except Exception as e:
        # skip unparseable structures
        continue

# batch graphs together for training
loader = DataLoader(pytorch_graphs, batch_size=16, shuffle=True)
print(f"Successfully processed {len(pytorch_graphs)} graphs into DataLoader.")

Converting SMILES into PyTorch graph tensors...
Successfully processed 200 graphs into DataLoader.


# 3. GNN Model Architecture

In [5]:
class DrugAffinityGNN(torch.nn.Module):
    def __init__(self, hidden_dim=64):
        super(DrugAffinityGNN, self).__init__()

        # input dim is 9 (atom features extracted by from_smiles)
        num_features = pytorch_graphs[0].num_node_features

        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = Linear(hidden_dim, 1) # final binding affinity score

    def forward(self, x, edge_index, batch):
        # 1. Message passing across molecular bonds
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        # 2. Pool atom representations into a single graph vector
        x = global_mean_pool(x, batch)

        # 3. Predict continuous binding affinity
        return self.lin(x)

# instantiate model and send to device
model = DrugAffinityGNN(hidden_dim=64).to(device)
print(model)

DrugAffinityGNN(
  (conv1): GCNConv(9, 64)
  (conv2): GCNConv(64, 64)
  (lin): Linear(in_features=64, out_features=1, bias=True)
)


# 4. Model Training

In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.MSELoss()

print("Training GNN...")
model.train()

for epoch in range(101):
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # make sure features are float
        out = model(batch.x.float(), batch.edge_index, batch.batch)
        loss = criterion(out.squeeze(), batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    # log progress periodically
    if epoch % 20 == 0:
        avg_loss = total_loss / len(loader.dataset)
        print(f"Epoch {epoch:>3} | MSE Loss: {avg_loss:.4f}")

print("Training complete!")

Training GNN...
Epoch   0 | MSE Loss: 66790265.6000
Epoch  20 | MSE Loss: 18120129.6800
Epoch  40 | MSE Loss: 18042333.4000
Epoch  60 | MSE Loss: 18153501.5200
Epoch  80 | MSE Loss: 18031252.4000
Epoch 100 | MSE Loss: 18165255.5200
Training complete!


# 5. Model Explainability with GNNExplainer

In [7]:
from torch_geometric.explain import Explainer, GNNExplainer

print("Configuring GNNExplainer...")
model.eval()

# setup explainer for graph-level regression
explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=200),
    explanation_type='model',
    node_mask_type='object', # score entire atoms
    model_config=dict(mode='regression', task_level='graph', return_type='raw')
)

# test drug: Imatinib (Gleevec)
target_smiles = "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CC=CC=C5"
test_graph = from_smiles(target_smiles).to(device)
test_graph.x = test_graph.x.float()

# single molecule needs a dummy batch index of zeros
dummy_batch = torch.zeros(test_graph.x.shape[0], dtype=torch.int64).to(device)

# calculate atom importance
explanation = explainer(test_graph.x, test_graph.edge_index, batch=dummy_batch)
atom_scores = explanation.node_mask.detach().cpu().numpy()

print(f"Finished explanation. Scored {len(atom_scores)} atoms.")

Configuring GNNExplainer...
Finished explanation. Scored 37 atoms.


# 6. 3D Heatmap

In [11]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from rdkit import Chem
from rdkit.Chem import AllChem
import py3Dmol

print("Building 3D visualization...")

mol = Chem.MolFromSmiles(target_smiles)
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, AllChem.ETKDG())
AllChem.MMFFOptimizeMolecule(mol)
mol_block = Chem.MolToMolBlock(mol)

scores = (atom_scores - atom_scores.min()) / (atom_scores.max() - atom_scores.min() + 1e-8)

# Switched to 'plasma' for a much more vibrant, glowing color palette on black
color_map = cm.get_cmap('plasma')

viewer = py3Dmol.view(width=800, height=500)
viewer.addModel(mol_block, 'mol')

# FIX 1: Brighter backbone (light silver) so the drug structure is clearly visible
viewer.setStyle({'model': -1}, {'stick': {'color': '#999999', 'radius': 0.15}})

for idx, score in enumerate(scores):
    if score > 0.4:
        pos = mol.GetConformer().GetAtomPosition(idx)
        hex_color = mcolors.to_hex(color_map(float(score)))

        viewer.addSphere({
            'center': {'x': pos.x, 'y': pos.y, 'z': pos.z},
            'radius': float(score) * 1.05, # FIX 2: Larger radius for a better "cloud" effect
            'color': hex_color,
            'alpha': 0.92 # FIX 3: Higher opacity so the bright colors aren't swallowed by the black background
        })

viewer.setBackgroundColor('black')
viewer.zoomTo()
viewer.show()

Building 3D visualization...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# 7. Exporting

In [12]:
from google.colab import files

# save weights dictionary
torch.save(model.state_dict(), 'drug_affinity_gnn.pth')
print("Model saved as drug_affinity_gnn.pth")

# download to your local drive
files.download('drug_affinity_gnn.pth')

Model saved as drug_affinity_gnn.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>